# Mount drive


In [2]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
%pip install -q "dask[dataframe]" pyarrow pandas tqdm

import os
import shutil
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import dask
import dask.dataframe as dd

from tqdm.auto import tqdm
from IPython.display import display


Mounted at /content/drive


# Convert zipped/tsv files into Parquet

In [5]:
%pip install -q pandas pyarrow tqdm


from pathlib import Path
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import zipfile
import csv
import time
from tqdm.auto import tqdm
import polars as pl
import gc

# CHANGE THIS PATH
FOLDER = Path("/content/drive/MyDrive/uc3m PhD/PhD Data/00 Raw data from PV/pv 2026-04 edition")

files = sorted(
    p for p in FOLDER.iterdir()
    if p.is_file()
    and p.name.lower().startswith(("g_", "pg_"))
    and p.name.lower().endswith((".tsv", ".zip"))
)

for file in tqdm(files, desc="Converting PatentsView"):

    name = file.name
    if name.lower().endswith(".zip"):
        name = name[:-4]
    if name.lower().endswith(".tsv"):
        name = name[:-4]

    output = FOLDER / f"{name}.parquet"

    if output.exists():
        print(f"Skipping {name}: already converted")
        continue

    start = time.time()
    print(f"\nProcessing {file.name}")

    try:
        # Detect ZIP archives even if their extension is .tsv
        if zipfile.is_zipfile(file):
            with zipfile.ZipFile(file) as z:
                members = [
                    n for n in z.namelist()
                    if n.lower().endswith(".tsv")
                    and not n.endswith("/")
                ]

                if len(members) != 1:
                    raise ValueError(
                        f"Expected one TSV, found {members}"
                    )

                with z.open(members[0]) as f:
                    df = pl.read_csv(
                        f,
                        separator="\t",
                        infer_schema_length=0
                    )
        else:
            df = pl.read_csv(
                file,
                separator="\t",
                infer_schema_length=0
            )

        rows, cols = df.shape

        df.write_parquet(
            output,
            compression="zstd"
        )

        del df
        gc.collect()

        print(
            f"Done: {name} | "
            f"{rows:,} rows | "
            f"{cols} columns | "
            f"{time.time()-start:.1f}s"
        )

    except Exception as e:
        print(f"ERROR: {file.name}: {e}")
        if "df" in locals():
            del df
        gc.collect()

print("\nConversion finished.")


Converting PatentsView:   0%|          | 0/18 [00:00<?, ?it/s]


Processing g_application.tsv.zip
Done: g_application | 9,451,902 rows | 6 columns | 7.2s

Processing g_assignee_disambiguated.tsv.zip
Done: g_assignee_disambiguated | 8,751,310 rows | 8 columns | 16.4s

Processing g_assignee_not_disambiguated.tsv.zip
Done: g_assignee_not_disambiguated | 8,751,310 rows | 8 columns | 21.7s

Processing g_cpc_at_issue.tsv.zip
Done: g_cpc_at_issue | 25,022,251 rows | 9 columns | 24.4s

Processing g_cpc_current.tsv.zip
Done: g_cpc_current | 59,805,669 rows | 7 columns | 43.5s

Processing g_patent.tsv.zip
Done: g_patent | 9,454,161 rows | 8 columns | 22.4s

Processing g_patent_abstract.tsv.zip
Done: g_patent_abstract | 9,454,161 rows | 2 columns | 157.0s

Processing g_persistent_assignee.tsv.zip
Done: g_persistent_assignee | 8,784,484 rows | 29 columns | 158.6s

Processing g_us_rel_doc.tsv.zip
Done: g_us_rel_doc | 16,914,186 rows | 9 columns | 22.6s

Processing pg_applicant_not_disambiguated.tsv.zip
Done: pg_applicant_not_disambiguated | 12,883,932 rows | 9 

# Load granted applications & get all applications into one DF

In [63]:
import pandas as pd

DRIVE_DIR = Path("/content/drive/MyDrive/uc3m PhD/PhD Data/00 Raw data from PV/pv 2026-04 edition")

# Load applications
pg_app = pd.read_parquet(DRIVE_DIR / "pg_published_application.parquet",
                     columns = ['pgpub_id', 'application_id', 'filing_date', 'published_date', 'filing_type']) # Identifier is pgpub_id (Pre-grant publication ID)

g_app = pd.read_parquet(DRIVE_DIR / "g_application.parquet", 
                     columns = ['patent_id', 'application_id', 'filing_date', 'patent_application_type']) # Identifier is patent_id (Granted/Issued Patent ID)

g_pat_withdrawn = pd.read_parquet(DRIVE_DIR / "g_patent.parquet", 
                     columns = ['patent_id', 'withdrawn']) # Identifier is patent_id (Granted/Issued Patent ID)

cw = pd.read_parquet(DRIVE_DIR / "pg_granted_pgpubs_crosswalk.parquet") # Crosswalk of publications to patents (to be used as a sense check)

# Print sizes
print(f"Before Shape of g_app:{g_app.shape}")
print(f"Before Shape of pg_app:{pg_app.shape}")

# Assert IDs for Pre-Grants
assert pg_app.set_index(["pgpub_id"]).index.is_unique == True, "Publication ID is NOT gisid"
assert pg_app.set_index(["application_id"]).index.is_unique == False, "Application ID is gisid"

# Keep the first pre-grant publication of each application
assert pg_app["published_date"].notna().all() == True, "Publication date is missing for some obs"
pg_app = pg_app.sort_values(["application_id", "published_date"], ascending = True)
pg_app = pg_app.drop_duplicates(subset="application_id", keep = "first")
assert pg_app.set_index(["application_id"]).index.is_unique == True, "Application ID is NOT gisid for pg_app"

# Assert IDs for Grants
assert g_app.set_index("patent_id").index.is_unique == True, "Patent ID is NOT gisid for g_app"
assert g_pat_withdrawn.set_index("patent_id").index.is_unique == True, "Patent ID is NOT gisid for g_pat_withdrawn"

g_app.set_index("patent_id", inplace = True)
g_pat_withdrawn.set_index("patent_id", inplace = True)

# Merge granted applications and patents and keep only not withdrawn patents
g_app = g_app.merge(g_pat_withdrawn, how="inner", left_index=True, right_index=True, validate="1:1", indicator=True)
g_app = g_app[g_app['withdrawn'] == "0"]
dup_g_app = g_app[g_app.duplicated('application_id', keep = False) == True]
dup_g_app = dup_g_app[[]]
g_app = g_app.merge(dup_g_app, how="left", left_index=True, right_index=True, validate="1:1", indicator="dup") # 6.4k patents have duplicated applications, these are mostly incorect entries for old years and a few design patents for the recent years, needs to be dropped
g_app = g_app[g_app.dup == "left_only"]
assert g_app.set_index("application_id").index.is_unique == True, "Application ID is NOT gisid for g_app"

# Print sizes
print(f"After Shape of g_app:{g_app.shape}")
print(f"After Shape of pg_app:{pg_app.shape}")

# Combine applications from pg_app and g_app
g_app.reset_index(inplace=True)
pg_app.reset_index(inplace=True)
g_app.set_index("application_id", inplace=True)
pg_app.set_index("application_id", inplace=True)

all_app = g_app.merge(pg_app, how="outer", left_index=True, right_index=True, validate="1:1", indicator="merge_g_gp")

# Manual check of columns of filing_date from pg_app and g_app shows 40k mismatches, and all the spots checks favor the g_app's column, so we drop the filing_date_y
all_app.drop("filing_date_y", axis = 1)
all_app.rename(columns  = {"filing_date_x":"filing_date"}, inplace = True)

# Compare patent_ids across cw and all_app
"""

Manual checks for 18k applications at df check again show that the patent_id column coming from g_app is accurate than those coming from cw
Therefore we drop the cw as a use-case from now on. 

"""
cw.set_index("application_id", inplace = True)
all_app_cw = all_app.merge(cw, how = "left", left_index=True, right_index=True, validate = "1:m", suffixes=("_apps", "_cw"), indicator="merge_apps_cw")
assert len(all_app_cw[all_app_cw.merge_apps_cw == "left_only"]) == 0, "There are some applications from all_app that did not merge to the crosswalk"
mask = all_app_cw["patent_id_apps"].notna() & all_app_cw["patent_id_cw"].notna()
check = all_app_cw.loc[
    mask & (all_app_cw["patent_id_apps"] != all_app_cw["patent_id_cw"])
] 

del mask, check, cw, g_app, pg_app, g_pat_withdrawn, all_app_cw

# Consolidate 
cols_to_keep = ['patent_id', 'filing_date', 'pgpub_id', 'published_date', 'filing_type']
all_app = all_app[cols_to_keep]
assert all_app.index.name == "application_id", "application_id is NOT the index of applications"
assert all_app.index.is_unique == True, "application_id is NOT gisid of applications"

# Dates 
all_app.filing_date = pd.to_datetime(all_app.filing_date, errors="coerce")
all_app.published_date = pd.to_datetime(all_app.published_date, errors="coerce")

# Sahpe of applications
print(f"Shape of all applications:{all_app.shape}")

# Save 
all_app.to_parquet("/content/drive/MyDrive/uc3m PhD/07 Continuous innovation space/02 Analysis/01 Main/01 Data load/out/01 All applications - app_id as index.parquet", index = True)


Before Shape of g_app:(9451902, 4)
Before Shape of pg_app:(8402596, 5)
After Shape of g_app:(9425979, 6)
After Shape of pg_app:(8382833, 5)
Shape of all applications:(12388750, 5)


# Combine applications with g_patent

In [3]:
# Load granted patents and merge with applications
DRIVE_DIR = Path("/content/drive/MyDrive/uc3m PhD/PhD Data/00 Raw data from PV/pv 2026-04 edition")

all_app = pd.read_parquet("/content/drive/MyDrive/uc3m PhD/07 Continuous innovation space/02 Analysis/01 Main/01 Data load/out/01 All applications - app_id as index.parquet") # uncomment if not loaded already
g_pat = pd.read_parquet(DRIVE_DIR / "g_patent.parquet") # Identifier is patent_id (Granted/Issued Patent ID)
g_pat.set_index("patent_id", inplace = True)
assert g_pat.index.is_unique == True
all_app = all_app.merge(g_pat, how = "outer", left_on = "patent_id", right_index=True, validate="m:1", indicator="merge_allapp_gpat")
all_app = all_app[(~all_app.patent_type.notna()) | (all_app.patent_type == "utility")] # For the granted patents, keep only utility patents
assert all_app.shape[0] == 11494732 # size check

# Check patents that were granted but did not get merged with applications
check = all_app[all_app.merge_allapp_gpat == "right_only"]
"""
Manual checks show that these are mostly withdrawn patents. There are above 6k patents that were granted and not withdrawn, but these did not get 
merged with any application. There are only handful of them after 2000 and most of them are pre-2000 era. The manual spot checks for these patents show 
that these are duplicated entires with the same application_id appearing g_app due to data errors, and since we have dropped these all from the applications, 
they are not merged to all_app. Since the number of patents is very small (i.e., 6k), we drop them from the granted publications as well,
and only keep those patents for which we can identify its filing information.
"""
all_app = all_app[all_app.merge_allapp_gpat != "right_only"]

# Create columns for granted pats
all_app["eventually_granted"] = all_app.merge_allapp_gpat == "both"

# Check that all granted patents have their patent_date
assert all_app['eventually_granted'].equals(all_app['patent_date'].notna())
assert all_app['eventually_granted'].equals(all_app['num_claims'].notna())

# Check that patent_date is always after filing_date
all_app["patent_date"] = pd.to_datetime(all_app['patent_date'], errors = "coerce")
assert all_app[all_app.patent_date < all_app.filing_date].shape[0] == 108 # Only 108 patents where filing_date is after the grant date
assert all_app[all_app.patent_date < all_app.filing_date].filing_date.dt.year.max() == 1999 #these are mostly old patents with data errors & can be dropped 
all_app = all_app[~(all_app.patent_date < all_app.filing_date)]

# Drop applications with no filing dates (there are only 34 of them pre 1980)
all_app = all_app[~((~all_app.filing_date.notna()) & all_app.eventually_granted)]
assert all_app.shape[0] == 11469409 # size check

del g_pat, check

## Load abstracts 

In [4]:
# Load publication abstracts
pub_abs = pd.read_parquet(DRIVE_DIR / "pg_published_application_abstract.parquet")
pub_abs.set_index("pgpub_id", inplace = True)
assert pub_abs.index.is_unique == True, "pgpub_id is NOT gisid for publication abstracts"

In [5]:
# Load granted patent abstracts
pat_abs = pd.read_parquet(DRIVE_DIR / "g_patent_abstract.parquet")
pat_abs.set_index("patent_id", inplace = True)
assert pat_abs.index.is_unique == True, "patent_id is NOT gisid for gratned patent abstracts"


## Merge applications with abstracts

In [6]:
# Merge abstracts of publications first as it takes priority over grant abstracts since it is the first version that got publicly available
all_app = all_app.merge(pub_abs, how="left", left_on = "pgpub_id", right_index=True, validate="m:1", indicator="merge_pgpub_abstract")
all_app = all_app.merge(pat_abs, how="left", left_on = "patent_id", right_index=True, validate="m:1", indicator="merge_pat_abstract")

del pub_abs, pat_abs 

all_app["consolidated_abstract"] = all_app['application_abstract'].fillna(all_app['patent_abstract']) # Take publication abstract first, then granted abtract if publication abstract is missing
all_app = all_app.dropna(subset=["consolidated_abstract"]) # Drop those applications/patents with no abstract at all
all_app["diff_abstract"] = (
    all_app["application_abstract"].ne(all_app["patent_abstract"])
    .mask(all_app[["application_abstract", "patent_abstract"]].isna().any(axis=1))
)# flag applications with differeing abstracts for publication and grant
all_app["diff_abstract"] = all_app["diff_abstract"].astype("boolean")


# Shape 
print(f"Applications with an abstract: {all_app.shape}")
all_app["consolidated_abstract"].isna().value_counts()

# Save 
all_app["patent_id"] = all_app["patent_id"].astype("string")
all_app["pgpub_id"] = all_app["pgpub_id"].astype("string")
all_app["filing_type"] = all_app["filing_type"].astype("string")
all_app["patent_type"] = all_app["patent_type"].astype("string")
all_app["patent_title"] = all_app["patent_title"].astype("string")
all_app["wipo_kind"] = all_app["wipo_kind"].astype("string")
all_app["filename"] = all_app["filename"].astype("string")
all_app["application_abstract"] = all_app["application_abstract"].astype("string")
all_app["consolidated_abstract"] = all_app["consolidated_abstract"].astype("string")
all_app["patent_abstract"] = all_app["patent_abstract"].astype("string")
all_app["num_claims"] = all_app["num_claims"].astype("Int64")
all_app["withdrawn"] = all_app["withdrawn"].astype("Int64")

all_app.to_parquet("/content/drive/MyDrive/uc3m PhD/07 Continuous innovation space/02 Analysis/01 Main/01 Data load/out/02 All applications with an abstract - app_id as index.parquet", index = True)



Applications with an abstract: (11469409, 20)
